# Scraper Broker Summary (Final)
Loop semua CSV di folder `Dataset/`, tambahkan kolom broker summary ke setiap file.

**Semua fix yang digabung di sini:**
1. `baca_tanggal_stockbit` & tabel di-scope ke `broker_section` — gak ketuker widget lain (mis. Running Trade) yang class-nya mirip.
2. `klik_previous` nunggu dinamis sampai tanggal beneran berubah, bukan `time.sleep()` flat.
3. `scrape_hari` nunggu tabel stabil (snapshot berturut-turut) alih-alih sleep flat.
4. Safety limit + debug print di loop navigasi.
5. Proteksi overwrite: row yang udah keisi valid gak akan ditimpa.
6. Validasi kode broker (2 huruf kapital) biar data ngasal dari tabel salah otomatis kebuang.
7. **Proteksi koneksi (baru)**: sebelum tiap baca tanggal / scrape hari, dicek dulu apakah Stockbit bisa diakses. Kalau internet putus / Stockbit lambat, script **pause otomatis** dan nunggu sampai pulih — TIDAK menganggapnya sebagai "tidak ada data", jadi tanggal yang beneran punya data gak akan salah ke-skip/ke-tandai libur gara-gara koneksi.

In [1]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import pandas as pd
import os
import glob
import time
import urllib.request
from datetime import datetime

In [ ]:
def scrape_broker_summary():

    DATASET_DIR      = "Dataset/"
    EDGE_DRIVER_PATH = "Driver/msedgedriver.exe"
    SELENIUM_PROFILE = r"D:\Tools\selenium_edge_profile"
    KOLOM_NON_BROKER = {"Date", "Close", "High", "Low", "Open", "Volume", "Sektor", "Sub_Sektor"}

    # ── Helper: cek koneksi LANGSUNG ke Stockbit (bukan cuma internet umum) ─
    #    Cek ke stockbit.com langsung, karena internet umum bisa hidup
    #    (mis. tes ke Google DNS sukses) padahal Stockbit sendiri yang lagi
    #    down/lambat/block sementara.
    def cek_koneksi(url="https://stockbit.com", timeout=5):
        try:
            req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "Mozilla/5.0"})
            urllib.request.urlopen(req, timeout=timeout)
            return True
        except Exception:
            return False

    def tunggu_koneksi_pulih(jeda_cek=5):
        if cek_koneksi():
            return
        print("  ⚠ Koneksi ke Stockbit terputus/lambat, menunggu pulih...")
        while not cek_koneksi():
            time.sleep(jeda_cek)
        print("  ✓ Koneksi ke Stockbit pulih, lanjut scraping...")

    # ── Helper: parse nilai '213.1B', '847.9M' → float miliar ──────────────
    def parse_value(val):
        val = str(val).replace(",", "").replace("+", "").strip()
        try:
            if "B" in val:   return float(val.replace("B", ""))
            elif "M" in val: return float(val.replace("M", "")) / 1000
            elif "K" in val: return float(val.replace("K", "")) / 1_000_000
            else:            return float(val)
        except Exception:    return 0.0

    # ── Helper: init Selenium Edge driver ───────────────────────────────────
    def init_driver():
        service = Service(EDGE_DRIVER_PATH)
        options = Options()
        options.add_argument(f"user-data-dir={SELENIUM_PROFILE}")
        options.add_argument("profile-directory=Stockbit")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)
        driver = webdriver.Edge(options=options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        return driver

    # ── Helper: ambil elemen root Broker Summary (di-query ulang tiap panggil,
    #    karena React kadang remount node → lebih aman daripada simpan referensi lama) ──
    def get_broker_section(driver):
        return driver.find_element(
            By.XPATH, "//p[contains(text(), 'Broker Summary')]/ancestor::div[2]"
        )

    # ── Helper: buka halaman saham & scroll ke Broker Summary ───────────────
    def buka_saham(driver, ticker):
        driver.get(f"https://stockbit.com/symbol/{ticker}")
        time.sleep(3)
        while True:
            try:
                el = WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.XPATH, "//p[contains(text(), 'Broker Summary')]"))
                )
                break
            except Exception:
                driver.execute_script("window.scrollBy(0, 500);")
                time.sleep(1)
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
        time.sleep(1)

    # ── Helper: baca tanggal, DISCOPE ke dalam broker_section ───────────────
    def baca_tanggal_stockbit(driver):
        try:
            broker_section = get_broker_section(driver)
            span = broker_section.find_element(By.CSS_SELECTOR, "span.guaywN")
            tgl_str = span.text.strip()  # contoh: '06 Jul 26'
            return datetime.strptime(tgl_str, "%d %b %y").strftime("%Y-%m-%d")
        except Exception:
            return None

    # ── Helper: klik tombol Previous day, tunggu DINAMIS sampai tanggal berubah ──
    def klik_previous(driver, timeout=10):
        tgl_sebelum = baca_tanggal_stockbit(driver)

        broker_section = get_broker_section(driver)
        prev_button = broker_section.find_element(
            By.XPATH, ".//button[@aria-label='Previous day']"
        )
        driver.execute_script("""
            const el = arguments[0];
            const rect = el.getBoundingClientRect();
            window.scrollBy(0, rect.top - 150);
        """, prev_button)
        time.sleep(0.2)
        driver.execute_script("arguments[0].click();", prev_button)

        # Tunggu sampai tanggal beneran berubah (atau timeout, biar gak infinite hang)
        akhir = time.time() + timeout
        while time.time() < akhir:
            tgl_sekarang = baca_tanggal_stockbit(driver)
            if tgl_sekarang is not None and tgl_sekarang != tgl_sebelum:
                return
            time.sleep(0.3)
        # kalau timeout, biarin lanjut — nanti divalidasi lagi di pemanggil

    # ── Helper: tunggu tabel broker stabil, DISCOPE ke broker_section ───────
    def tunggu_tabel_stabil(driver, broker_section, timeout=15, jeda_cek=0.4, stabil_selama=2):
        akhir = time.time() + timeout
        snapshot_sebelum = None
        stabil_count = 0

        while time.time() < akhir:
            rows = broker_section.find_elements(By.CSS_SELECTOR, "tbody.ant-table-tbody tr.ant-table-row")
            if not rows:
                time.sleep(jeda_cek)
                continue

            try:
                snapshot = (len(rows), rows[0].text, rows[-1].text)
            except Exception:
                time.sleep(jeda_cek)
                continue

            if snapshot == snapshot_sebelum:
                stabil_count += 1
                if stabil_count >= stabil_selama:
                    return rows
            else:
                stabil_count = 0

            snapshot_sebelum = snapshot
            time.sleep(jeda_cek)

        return broker_section.find_elements(By.CSS_SELECTOR, "tbody.ant-table-tbody tr.ant-table-row")

    # ── Helper: scrape tabel broker hari ini, DISCOPE ke broker_section ─────
    def scrape_hari(driver):
        broker_section = get_broker_section(driver)
        try:
            WebDriverWait(driver, 15).until(
                lambda d: len(broker_section.find_elements(By.CSS_SELECTOR, "tbody.ant-table-tbody tr.ant-table-row")) > 0
            )
        except Exception:
            return None

        rows = tunggu_tabel_stabil(driver, broker_section)
        buy, sell = {}, {}
        for row in rows:
            try:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) < 8: continue
                bb, vb = cols[0].text.strip(), cols[1].text.strip()
                bs, vs = cols[4].text.strip(), cols[5].text.strip()
            except Exception:
                continue
            if bb and bb != "-" and len(bb) == 2:
                buy[bb]  = buy.get(bb, 0)  + parse_value(vb)
            if bs and bs != "-" and len(bs) == 2:
                sell[bs] = sell.get(bs, 0) + parse_value(vs)
        if not buy and not sell:
            return None

        # Validasi kode broker: harus 2 huruf kapital, bukan data ngasal dari tabel lain
        buy  = {b: v for b, v in buy.items()  if b.isalpha() and b.isupper()}
        sell = {b: v for b, v in sell.items() if b.isalpha() and b.isupper()}
        if not buy and not sell:
            return None

        brokers = set(buy) | set(sell)
        return {b: round(buy.get(b, 0) - sell.get(b, 0), 4) for b in brokers}

    # ── Helper: simpan satu baris ke CSV, DENGAN PROTEKSI OVERWRITE ─────────
    def simpan(df, csv_path, date_str, data):
        for broker in data:
            if broker not in df.columns:
                df[broker] = float("nan")
                brok_cols = [c for c in df.columns if c not in KOLOM_NON_BROKER]
                sudah = df[brok_cols].drop(columns=[broker]).notna().any(axis=1)
                df.loc[sudah, broker] = 0.0

        mask = df["Date"] == date_str
        brok_cols = [c for c in df.columns if c not in KOLOM_NON_BROKER]

        # PROTEKSI: kalau row ini udah punya minimal 1 kolom broker terisi, jangan ditimpa
        if brok_cols:
            sudah_terisi = df.loc[mask, brok_cols].notna().any(axis=1)
            if sudah_terisi.any():
                print(f"    [proteksi] {date_str} sudah ada data, dilewati (tidak ditimpa)")
                return df

        for broker, val in data.items():
            df.loc[mask, broker] = val

        for broker in brok_cols:
            if broker not in data:
                df.loc[mask, broker] = 0.0

        df.to_csv(csv_path, index=False)
        return df

    # ── Main logic ──────────────────────────────────────────────────────────
    semua_csv = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))
    if not semua_csv:
        print("Tidak ada file CSV di Dataset/")
        return

    print(f"Total ticker: {len(semua_csv)}")

    tunggu_koneksi_pulih()  # pastikan koneksi hidup sebelum buka browser
    driver = init_driver()

    MAX_KLIK_NAVIGASI = 500

    try:
        driver.get("https://stockbit.com/login")
        print("Browser terbuka. Pastikan sudah login.")
        time.sleep(3)

        for idx, csv_path in enumerate(semua_csv):
            filename  = os.path.basename(csv_path)
            ticker    = filename.replace(".csv", "")
            ticker_sb = ticker.replace(".JK", "") if ticker.endswith(".JK") else ticker

            print(f"\n[{idx+1}/{len(semua_csv)}] {ticker} → buka '/symbol/{ticker_sb}'")

            df        = pd.read_csv(csv_path)
            brok_cols = [c for c in df.columns if c not in KOLOM_NON_BROKER]

            if brok_cols:
                belum_mask = df[brok_cols].isna().all(axis=1)
            else:
                belum_mask = pd.Series([True] * len(df), index=df.index)

            if not belum_mask.any():
                print("  → Sudah lengkap, skip")
                continue

            tanggal_csv     = set(df["Date"].tolist())
            tanggal_pertama = df["Date"].min()
            tanggal_target  = df.loc[belum_mask, "Date"].max()
            jumlah_belum    = belum_mask.sum()

            print(f"  {jumlah_belum} tanggal belum di-scrape, mulai dari {tanggal_target}")

            tunggu_koneksi_pulih()

            try:
                buka_saham(driver, ticker_sb)
            except Exception as e:
                print(f"  ✗ Gagal buka halaman: {e}")
                continue

            # Navigasi pakai < sampai ketemu tanggal_target, dengan safety limit
            print(f"  Navigasi ke {tanggal_target}...")
            klik_ke = 0
            gagal_navigasi = False
            while True:
                tunggu_koneksi_pulih()
                tgl_sb = baca_tanggal_stockbit(driver)

                if tgl_sb is None:
                    klik_previous(driver)
                    klik_ke += 1
                elif tgl_sb == tanggal_target:
                    break
                elif tgl_sb < tanggal_pertama:
                    print(f"  ✗ Tanggal target tidak ditemukan, skip ticker ini")
                    gagal_navigasi = True
                    break
                else:
                    klik_previous(driver)
                    klik_ke += 1

                if klik_ke % 20 == 0 and klik_ke > 0:
                    print(f"    [debug] posisi navigasi sekarang: {tgl_sb} ({klik_ke} klik)")

                if klik_ke > MAX_KLIK_NAVIGASI:
                    print(f"  ✗ Kelewat batas {MAX_KLIK_NAVIGASI} klik saat navigasi, skip ticker ini")
                    gagal_navigasi = True
                    break

            if gagal_navigasi:
                continue

            if baca_tanggal_stockbit(driver) != tanggal_target:
                continue

            # Loop scraping mundur
            scraped = 0
            klik_ke = 0
            while True:
                tunggu_koneksi_pulih()  # ← proteksi utama: pastikan koneksi hidup SEBELUM baca/scrape
                tgl_sb = baca_tanggal_stockbit(driver)

                if tgl_sb is None:
                    print("  ✗ Gagal baca tanggal, coba lanjut")
                    klik_previous(driver)
                    klik_ke += 1
                    if klik_ke > MAX_KLIK_NAVIGASI:
                        print(f"  ✗ Kelewat batas klik saat scraping, hentikan ticker ini")
                        break
                    continue

                if tgl_sb < tanggal_pertama:
                    print(f"  Selesai (melewati tanggal awal CSV)")
                    break

                if tgl_sb not in tanggal_csv:
                    klik_previous(driver)
                    continue

                mask_tgl = df["Date"] == tgl_sb
                brok_cols = [c for c in df.columns if c not in KOLOM_NON_BROKER]
                if brok_cols and not df.loc[mask_tgl, brok_cols].isna().all(axis=1).any():
                    klik_previous(driver)
                    continue

                try:
                    tunggu_koneksi_pulih()  # cek lagi tepat sebelum scrape, jaga-jaga koneksi drop di antara baca tanggal & scrape
                    data = scrape_hari(driver)
                    if data is None:
                        print(f"  → {tgl_sb} tidak ada data (kemungkinan libur bursa)")
                    else:
                        df = simpan(df, csv_path, tgl_sb, data)
                        brok_cols = [c for c in df.columns if c not in KOLOM_NON_BROKER]
                        scraped += 1
                        print(f"  ✓ {tgl_sb} — {len(data)} broker ({scraped}/{jumlah_belum})")
                except Exception as e:
                    print(f"  ✗ Gagal scrape {tgl_sb}: {e}")

                klik_previous(driver)

    finally:
        driver.quit()
        print("\nSelesai. Browser ditutup.")


scrape_broker_summary()

Total ticker: 775
Browser terbuka. Pastikan sudah login.

[1/775] AADI.JK → buka '/symbol/AADI'
  5 tanggal belum di-scrape, mulai dari 2026-07-24
  Navigasi ke 2026-07-24...
  ✓ 2026-07-24 — 45 broker (1/5)
  ✓ 2026-07-23 — 45 broker (2/5)
  ✓ 2026-07-22 — 42 broker (3/5)
  ✓ 2026-07-21 — 46 broker (4/5)
  ✓ 2026-07-20 — 44 broker (5/5)
  Selesai (melewati tanggal awal CSV)

[2/775] AALI.JK → buka '/symbol/AALI'
  5 tanggal belum di-scrape, mulai dari 2026-07-24
  Navigasi ke 2026-07-24...
  ✓ 2026-07-24 — 35 broker (1/5)
  ✓ 2026-07-23 — 36 broker (2/5)
  ✓ 2026-07-22 — 32 broker (3/5)
  ✓ 2026-07-21 — 32 broker (4/5)
  ✓ 2026-07-20 — 36 broker (5/5)
  Selesai (melewati tanggal awal CSV)

[3/775] ABBA.JK → buka '/symbol/ABBA'
  10 tanggal belum di-scrape, mulai dari 2026-07-24
  Navigasi ke 2026-07-24...
  ✗ Tanggal target tidak ditemukan, skip ticker ini

[4/775] ABDA.JK → buka '/symbol/ABDA'
  5 tanggal belum di-scrape, mulai dari 2026-07-24
  Navigasi ke 2026-07-24...
  ✓ 2026-07-